In [2]:
def transitive_relation(x,c):
    return x+c
    
def get_estimate_c(full_dfp7_goodgr_small, passages_fit = [0,1,2]):
    xs=[]
    ys=[]
    xs_min = []
    xs_max = []
    passages=[]
    xsfit = []
    ysfit = []
    for passage in [0,1,2,3,4,5,6,7]:
        passage_df = full_dfp7_goodgr_small.loc[full_dfp7_goodgr_small['passage']==passage,:]
        if len(passage_df['parent_subjects'].unique())<3:
            continue
        x1 = passage_df.loc[passage_df['parent_subjects']=='AA-AE','boot_med1_shift'].values[0]
        x2 =passage_df.loc[passage_df['parent_subjects']=='AE-AF','boot_med1_shift'].values[0]
        xs.append(x1+x2)
        ys.append(passage_df.loc[passage_df['parent_subjects']=='AA-AF','boot_med1_shift'].values[0])
        passages.append(passage)
        if passage in passages_fit:
            
            xsfit.append(x1+x2)
            ysfit.append(passage_df.loc[passage_df['parent_subjects']=='AA-AF','boot_med1_shift'].values[0])
    if len(ys)==0:
        return np.nan,np.nan,np.nan,np.nan
    if len(ysfit)==0:
        return np.nan,np.nan,np.nan,np.nan
    c,_ = curve_fit(transitive_relation,xsfit,ysfit)
  #  print(curve_fit(transitive_relation,xs,ys))
   # print(c)
    return c[0],xs, ys,passages

In [3]:

def adjust_freq(df_both,shift=True, thresh=1e-3,):
    e003_metadata = pd.read_csv('e003_coalescence_metadata_round4_good.csv').set_index('sample')
    df_both = df_both.loc[np.intersect1d(df_both.index.values,e003_metadata.index.values),:]
    df_both=df_both.loc[df_both['total_shift']<.1,:]

    df_both_meta = pd.concat([df_both,e003_metadata.loc[np.intersect1d(df_both.index.values,
                                                                                   e003_metadata.index.values),:]],
                                         axis=1).reset_index()

    df_both_meta_good=df_both_meta.loc[df_both_meta['total_shift']<.1,:]
    for col in ['boot_med1','boot_low1','boot_high1','boot_med2','boot_low2','boot_high2']:
        df_both_meta_good.loc[df_both_meta_good[col]<thresh,col]=thresh
        df_both_meta_good.loc[df_both_meta_good[col]>1-thresh,col]=1-thresh
    if shift:
        df_both_meta_good['boot_med1_shift']=df_both_meta_good['boot_med1']/(df_both_meta_good['boot_med1']+df_both_meta_good['boot_med2'])
        df_both_meta_good['boot_high1_shift']=df_both_meta_good['boot_high1']/(df_both_meta_good['boot_high1']+df_both_meta_good['boot_high2'])
        df_both_meta_good['boot_low1_shift']=df_both_meta_good['boot_low1']/(df_both_meta_good['boot_low1']+df_both_meta_good['boot_low2'])
        for col in ['boot_med1_shift','boot_low1_shift','boot_high1_shift']:
            df_both_meta_good.loc[df_both_meta_good[col]<thresh,col]=thresh
            df_both_meta_good.loc[df_both_meta_good[col]>1-thresh,col]=1-thresh
    df_both_meta_good_logit = df_both_meta_good.copy()
    df_both_meta_good_logit[['boot_med1','boot_low1','boot_high1',
    'boot_med2','boot_low2','boot_high2',]] = np.log(df_both_meta_good_logit[['boot_med1','boot_low1','boot_high1',
    'boot_med2','boot_low2','boot_high2',]]/(1-df_both_meta_good_logit[['boot_med1','boot_low1','boot_high1',
    'boot_med2','boot_low2','boot_high2',]]) )
    if shift:
        df_both_meta_good_logit[['boot_med1_shift','boot_low1_shift','boot_high1_shift']] = \
            np.log(df_both_meta_good_logit[['boot_med1_shift','boot_low1_shift','boot_high1_shift']]/(1-\
             df_both_meta_good_logit[['boot_med1_shift','boot_low1_shift','boot_high1_shift']]))

    return df_both_meta_good_logit

In [4]:
bad_inos = ['100146-AA-AF-mBHI',
 '100146-AE-AF-mBHI',
 '102478-AA-AF-mBHI',
 '102478-AE-AF-mBHI',
 '100196-AA-AF-mBHI',
 '100196-AE-AF-mBHI',
 '100196-AA-AE-mGAM',
 '100196-AE-AF-mGAM',
 '102544-AA-AF-mBHI',
 '102544-AE-AF-mBHI',
 '101349-AA-AF-mBHI',
 '101349-AE-AF-mBHI',
 '102506-AA-AE-mBHI',
 '102506-AE-AF-mBHI',
 '102506-AA-AF-mBHI',
 '102506-AE-AF-mBHI',
 '102506-AA-AE-mGAM',
 '102506-AE-AF-mGAM',
 '101346-AA-AE-mBHI',
 '101346-AE-AF-mBHI',
 '101346-AA-AF-mBHI',
 '101346-AE-AF-mBHI',
 '101346-AA-AE-mGAM',
 '101346-AE-AF-mGAM',
 '101346-AA-AE-mBHI',
 '101346-AA-AF-mBHI',
 '100120-AA-AF-mBHI',
 '100120-AE-AF-mBHI',
 '102327-AA-AF-mBHI',
 '102327-AE-AF-mBHI']

all_dfs=[]
e003_metadata = pd.read_csv('e003_coalescence_metadata_round4_good.csv').set_index('sample')
species_list = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/*/')
species_list =[sp.split('/')[-2] for sp in species_list]
for sp in species_list:
    for ino in ['AA-AE-mBHI','AA-AF-mBHI','AE-AF-mBHI',
                'AA-AE-mGAM','AA-AF-mGAM','AE-AF-mGAM',]:
        fname = f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3/{sp}/{ino}_parent1_info.csv'
        if f'{sp}-{ino}' in bad_inos:
            continue
        if os.path.exists(fname):
            df_both = get_both_dfs(fname)
            df=adjust_freq(df_both).reset_index()
            df['species_id']=sp
            all_dfs.append(df)
            
all_dfs=pd.concat(all_dfs)
all_dfs_adjust= all_dfs.copy()
all_dfs_gr = all_dfs.groupby(['species_id','parent_subjects','type_mesocosm','passage']).median(numeric_only=True).reset_index()
full_dfp7=all_dfs.copy()
full_dfp7['species-type_meso']= full_dfp7['species_id'] + '-' + full_dfp7['type_mesocosm']
full_dfp7['species-env']= full_dfp7['species_id']+'-' + full_dfp7['parent_media'] + '-' + full_dfp7['media'] 
id_columns = ['species-type_meso','species-env','parent_subjects', 'media', 'parent_media', 'species_id','type_mesocosm','species',
              'inoculumn_sample','inoculumn',
             'passage']
full_dfp7 = full_dfp7.loc[~full_dfp7['boot_med1_shift'].isna(),:]    

full_dfp7_goodgr = full_dfp7[id_columns + ['boot_med1_shift',]].groupby(id_columns).median().reset_index()
full_dfp7_goodgrmax = full_dfp7[id_columns + ['boot_med1_shift',]].groupby(id_columns).max().reset_index()
full_dfp7_goodgrmin = full_dfp7[id_columns + ['boot_med1_shift',]].groupby(id_columns).min().reset_index()

full_dfp7_goodgr['sp-inenv']=full_dfp7_goodgr['species_id']+'-'+full_dfp7_goodgr['parent_media']
full_dfp7_goodgrmin['sp-inenv']=full_dfp7_goodgrmin['species_id']+'-'+full_dfp7_goodgrmin['parent_media']
full_dfp7_goodgrmax['sp-inenv']=full_dfp7_goodgrmax['species_id']+'-'+full_dfp7_goodgrmax['parent_media']

In [5]:

sp_envs=[]
c_ests=[]
big_df = []

for sp_env in full_dfp7_goodgr['species-env'].unique():
    
    full_dfp7_goodgr_small = full_dfp7_goodgr.loc[full_dfp7_goodgr['species-env']==sp_env,:]
    inenv = sp_env.split('-')[0]+'-' + sp_env.split('-')[1]
    ins = full_dfp7_goodgr.loc[(full_dfp7_goodgr['passage']==0)*(full_dfp7_goodgr['sp-inenv']==inenv),:]
    if full_dfp7_goodgr_small['passage'].min()>0:
        full_dfp7_goodgr_small=pd.concat([full_dfp7_goodgr_small,ins])
        
    insmin = full_dfp7_goodgrmin.loc[(full_dfp7_goodgrmin['passage']==0)*(full_dfp7_goodgrmin['sp-inenv']==inenv),:]   
    full_dfp7_goodgr_smallmin = full_dfp7_goodgrmin.loc[full_dfp7_goodgrmin['species-env']==sp_env,:]
    if full_dfp7_goodgr_smallmin['passage'].min()>0:
        full_dfp7_goodgr_smallmin=pd.concat([full_dfp7_goodgr_smallmin,insmin])

    insmax = full_dfp7_goodgrmax.loc[(full_dfp7_goodgrmax['passage']==0)*(full_dfp7_goodgrmax['sp-inenv']==inenv),:] 
    full_dfp7_goodgr_smallmax = full_dfp7_goodgrmax.loc[full_dfp7_goodgrmax['species-env']==sp_env,:]
    if full_dfp7_goodgr_smallmax['passage'].min()>0:
        full_dfp7_goodgr_smallmax=pd.concat([full_dfp7_goodgr_smallmax,insmax])

    if len(full_dfp7_goodgr_small['parent_subjects'].unique())==3:
        c_est,xs,ys,passages = get_estimate_c(full_dfp7_goodgr_small,passages_fit = [0,1])
        c_estmin,xs_min,ys,passages = get_estimate_c(full_dfp7_goodgr_smallmin,passages_fit = [0,1])
        c_estmin,xs_max,ys,passages = get_estimate_c(full_dfp7_goodgr_smallmax,passages_fit = [0,1])
        if np.isnan(c_est):
            continue
        sp_envs.append(sp_env)
        c_ests.append(c_est)
        yests = xs + c_est
        rmse = np.sqrt(np.sum((yests-ys)**2)/len(ys))
    
        df=pd.DataFrame(data={'species-env':sp_env, 'c_ests':c_est, 'xs':xs,'ys':ys,
                              'xs_min': xs_min,  'xs_max': xs_max, 
                              'rmse':rmse,
                              'passages':passages})
        big_df.append(df)
   # print(c_est)

   # print(sp_env)


c_ests_df = pd.DataFrame(data={'species-env':sp_envs, 'c_ests':c_ests})
big_df = pd.concat(big_df)
big_df.head()
print(len(big_df))

115


/var/folders/m8/d6y7bwh127db53wc5fyr9w6c0000gn/T/ipykernel_3692/2059199595.py:29: OptimizeWarning: Covariance of the parameters could not be estimated
  c,_ = curve_fit(transitive_relation,xsfit,ysfit)


In [6]:
big_df['ys_est']=big_df['xs']+big_df['c_ests']
big_df['ys_estmin']=big_df['xs_min']+big_df['c_ests']
big_df['ys_estmax']=big_df['xs_max']+big_df['c_ests']

thresh=1e-3
full_dfp7.loc[full_dfp7['boot_med1_shift']<np.log(thresh/(1-thresh)), 'boot_med1_shift'] = np.log(thresh/(1-thresh))
big_df.loc[big_df['ys_est']<np.log(thresh/(1-thresh)),'ys_est']=np.log(thresh/(1-thresh))
big_df.loc[big_df['ys_est']>np.log((1-thresh)/(thresh)),'ys_est']=np.log((1-thresh)/(thresh))

big_df.loc[big_df['ys_estmin']<np.log(thresh/(1-thresh)),'ys_estmin']=np.log(thresh/(1-thresh))
big_df.loc[big_df['ys_estmin']>np.log((1-thresh)/(thresh)),'ys_estmin']=np.log((1-thresh)/(thresh))

big_df.loc[big_df['ys_estmax']<np.log(thresh/(1-thresh)),'ys_estmax']=np.log(thresh/(1-thresh))
big_df.loc[big_df['ys_estmax']>np.log((1-thresh)/(thresh)),'ys_estmax']=np.log((1-thresh)/(thresh))
scatter = hv.Scatter(big_df.loc[big_df['passages']>1,:], kdims =  'ys_est', vdims=['ys','passages']).opts(alpha = 1.0, #height=700,
                                                                          xlim=(-7.5,7.5),ylim = (-7.5,7.5),
                                                                                           width = 400,line_color='black',             
                                                                                      cmap = bokeh.palettes.Bokeh[6], colorbar=True,
                                                                                 color='passages',
                                                                                 legend_position='right',  size=10,
                                                                                  clabel='Passage',
                                                                              xlabel='Pred Freq (Transitivity)',ylabel='Actual Freq',)
exps_adjust = np.array([.001,.01,.1,.5,.9,.99,.999])
    
lins_adjust = np.log(exps_adjust/(1-exps_adjust))

scatter.opts(xticks=[(lins_adjust[i], exps_adjust[i]) for i in range(len(lins_adjust))],
                 yticks=[(lins_adjust[i], exps_adjust[i]) for i in range(len(lins_adjust))]
                )

scatter
p = hv.render(scatter)
p.xaxis.axis_label = 'Pred Freq (Transitive)'
p.yaxis.axis_label = 'Obs Freq'
bokeh.io.show(p)
p.output_backend='svg'
export_plot_pdf(p, 'trans_scatter')

In [7]:
big_df['ys_est']=big_df['xs']+big_df['c_ests']
big_df['ys_estmin']=big_df['xs_min']+big_df['c_ests']
big_df['ys_estmax']=big_df['xs_max']+big_df['c_ests']

thresh=1e-3
full_dfp7.loc[full_dfp7['boot_med1_shift']<np.log(thresh/(1-thresh)), 'boot_med1_shift'] = np.log(thresh/(1-thresh))
big_df.loc[big_df['ys_est']<np.log(thresh/(1-thresh)),'ys_est']=np.log(thresh/(1-thresh))
big_df.loc[big_df['ys_est']>np.log((1-thresh)/(thresh)),'ys_est']=np.log((1-thresh)/(thresh))

big_df.loc[big_df['ys_estmin']<np.log(thresh/(1-thresh)),'ys_estmin']=np.log(thresh/(1-thresh))
big_df.loc[big_df['ys_estmin']>np.log((1-thresh)/(thresh)),'ys_estmin']=np.log((1-thresh)/(thresh))

big_df.loc[big_df['ys_estmax']<np.log(thresh/(1-thresh)),'ys_estmax']=np.log(thresh/(1-thresh))
big_df.loc[big_df['ys_estmax']>np.log((1-thresh)/(thresh)),'ys_estmax']=np.log((1-thresh)/(thresh))


big_df['ys_est_linear'] = 1/(1+np.exp(-big_df['ys_est']))
big_df['ys_estmin_linear'] = 1/(1+np.exp(-big_df['ys_estmin']))
big_df['ys_estmax_linear'] = 1/(1+np.exp(-big_df['ys_estmax']))
big_df['ys_linear'] = 1/(1+np.exp(-big_df['ys']))
scatter = hv.Scatter(big_df.loc[big_df['passages']>1,:], kdims =  'ys_est_linear', vdims=['ys_linear','passages']).opts(alpha = 1.0, #height=700,
                                                                          xlim=(-0.05,1.05),ylim = (-0.05,1.05),
                                                                                           width = 400,line_color='black',             
                                                                                      cmap = bokeh.palettes.Bokeh[6], colorbar=True,
                                                                                 color='passages',
                                                                                 legend_position='right',  size=10,
                                                                                  clabel='Passage',
                                                                              xlabel='Pred Freq (Transitivity)',ylabel='Actual Freq',)


scatter
p = hv.render(scatter)
p.xaxis.axis_label = 'Pred Freq (Transitive)'
p.yaxis.axis_label = 'Obs Freq'
bokeh.io.show(p)
p.output_backend='svg'
export_plot_pdf(p, 'trans_scatter_linear')

In [8]:
big_df['error']=big_df['ys_linear']-big_df['ys_est_linear']
print(len(big_df['species-env'].unique()))

18


In [9]:
frequencies, edges = np.histogram(big_df['error'].values,bins=20)
#print('Values: %s, Edges: %s' % (frequencies.shape[0], edges.shape[0]))
p=hv.Histogram((edges, frequencies)).opts(width=400, xlabel='Obs Freq - Pred Freq', 
                                        ylabel='Counts',fill_color= bokeh.palettes.Bright[6][1]
                                       )
p=hv.render(p)
p.yaxis.visible = False

p.output_backend='svg'
p.xaxis.axis_label_text_font_size='20px'
p.xaxis.major_label_text_font_size='15px'
bokeh.io.show(p)
export_plot_pdf(p, 'trans_error')

In [15]:
import os
inos=['AE-AF-mBHI','AA-AF-mBHI','AA-AE-mBHI',]
inos = ['AE-AF-mGAM','AA-AF-mGAM','AA-AE-mGAM']+['AE-AF-mBHI','AA-AF-mBHI','AA-AE-mBHI',]

#inos = ['AA-AF-mGAM']
1#inos = ['AE-AF-mBHI']
species = [102438,100196, 100099, 100146, 101346, 102506, 102478]
species = [102438,100196, 100099, 100146, 101346, 102506, 102478,102358,101294]
species = [101294]
inos=['AA-AF-mGAM','AA-AF-mBHI']
for sp in species:
    plots_all=[]
    for ino in inos:
        fname = f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/{sp}/{ino}_parent1_info.csv'
        if f'{sp}-{ino}' in bad_inos:
            continue
        if not os.path.isfile(fname):
            print('oo')
            continue
        psstrain,type_mesos = make_logit_plot(sp, ino,
                                       colorby='const',
                                        )
      #  print(psstrain)
        for i in range(len(psstrain)):
            p1 = psstrain[i]
            meso = type_mesos[i]
            med1=meso.split('-')[-2]
            med2 = meso.split('-')[-1]
            ino1 = f'{sp}-{med1}-{med2}'
            p1.legend.visible=False
            preds = big_df.loc[big_df['species-env']==ino1,:]
          #  print(preds)
            if len(preds)>0:
                p1.line(preds['passages'],preds['ys_est'],color='grey')
                p1.circle(preds['passages'],preds['ys_est'],color='grey',size=5)
                p1.line(preds['passages'],preds['ys_estmin'],color='grey',line_dash='dashed')
                p1.line(preds['passages'],preds['ys_estmax'],color='grey',line_dash='dashed')
                p1.output_backend = "svg"

                p2s,_= make_logit_plot(sp, f'AA-AE-{med1}',
                                       colorby='const',
                                         medias = [med2]
                                        )
                p2=p2s[0]
                p3s,_= make_logit_plot(sp, f'AE-AF-{med1}',
                                       colorby='const',
                                         medias = [med2]
                                        )
                p3=p3s[0]
                p2.legend.visible=False
                p1.legend.visible=False
                p3.legend.visible=False
                p2.output_backend = "svg"
                p3.output_backend='svg'
                grid = bokeh.layouts.gridplot([p2,p3,p1],ncols=3)
                bokeh.io.show(grid)
                fname_prefix = f'{sp}_{ino}_{med2}_transitive'

                plot_fname_svg=f'plots/{fname_prefix}.svg'
                plot_fname_pdf=f'plots/{fname_prefix}.pdf'
                bokeh.io.export_svg(grid,filename=plot_fname_svg)
                convert_scripts = f'rsvg-convert -f pdf -o {plot_fname_pdf} {plot_fname_svg}' 
                os.system(convert_scripts)

                

    

oo


In [16]:
import os
inos=['AE-AF-mBHI','AA-AF-mBHI','AA-AE-mBHI',]
inos = ['AE-AF-mGAM','AA-AF-mGAM','AA-AE-mGAM']+['AE-AF-mBHI','AA-AF-mBHI','AA-AE-mBHI',]

#inos = ['AA-AF-mGAM']
1#inos = ['AE-AF-mBHI']
species = [102438,100196, 100099, 100146, 101346, 102506, 102478]
species = [102438,100196, 100099, 100146, 101346, 102506, 102478,102358,101294]
species = [102438,100099,101294]
inos=['AA-AF-mGAM','AA-AF-mBHI']
for sp in species:
    plots_all=[]
    for ino in inos:
        fname = f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_sel_bootstrapv3/{sp}/{ino}_parent1_info.csv'
        if f'{sp}-{ino}' in bad_inos:
            continue
        if not os.path.isfile(fname):
            print('oo')
            continue
        psstrain,type_mesos = make_logit_plot(sp, ino,
                                       colorby='const',logit=False,
                                        )
      #  print(psstrain)
        for i in range(len(psstrain)):
            p1 = psstrain[i]
            meso = type_mesos[i]
            med1=meso.split('-')[-2]
            med2 = meso.split('-')[-1]
            ino1 = f'{sp}-{med1}-{med2}'
            p1.legend.visible=False
            preds = big_df.loc[big_df['species-env']==ino1,:]
            preds['ys_est_linear'] = 1/(1+np.exp(-preds['ys_est']))
            preds['ys_estmin_linear'] = 1/(1+np.exp(-preds['ys_estmin']))
            preds['ys_estmax_linear'] = 1/(1+np.exp(-preds['ys_estmax']))
                      #  print(preds)
            if len(preds)>0:
                p1.line(preds['passages'],preds['ys_est_linear'],color='grey')
                p1.circle(preds['passages'],preds['ys_est_linear'],color='grey',size=5)
                p1.line(preds['passages'],preds['ys_estmin_linear'],color='grey',line_dash='dashed')
                p1.line(preds['passages'],preds['ys_estmax_linear'],color='grey',line_dash='dashed')
                p1.output_backend = "svg"

                p2s,_= make_logit_plot(sp, f'AA-AE-{med1}',
                                       colorby='const',
                                         medias = [med2],logit=False,
                                        )
                p2=p2s[0]
                p3s,_= make_logit_plot(sp, f'AE-AF-{med1}',
                                       colorby='const',
                                         medias = [med2],logit=False,
                                        )
                if len(p3s)==0:
                    continue
                p3=p3s[0]
                p2.legend.visible=False
                p1.legend.visible=False
                p3.legend.visible=False
                for p in [p1,p2,p3]:
                    p.yaxis.axis_label_text_font_style = 'normal'
                    p.xaxis.axis_label_text_font_style = 'normal'
                    p.legend.visible=False
                p2.output_backend = "svg"
                p3.output_backend='svg'
                
                grid = bokeh.layouts.gridplot([p2,p3,p1],ncols=3)
                bokeh.io.show(p1)
                fname_prefix = f'{sp}_{ino}_{med2}_transitive_linear'

                plot_fname_svg=f'plots/{fname_prefix}.svg'
                plot_fname_pdf=f'plots/{fname_prefix}.pdf'
                bokeh.io.export_svg(grid,filename=plot_fname_svg)
                convert_scripts = f'rsvg-convert -f pdf -o {plot_fname_pdf} {plot_fname_svg}' 
                os.system(convert_scripts)

                

    

oo


In [12]:
bokeh.palettes.Set2[8][2]

'#8da0cb'